# 🔬 AA-CLIP: Reproduce Paper — Kaggle Notebook (CVPR 2025)

## 🎯 Mục tiêu: Tái tạo kết quả Zero-Shot Anomaly Detection

### Cross-Dataset Protocol (theo Section 4.1 trong paper)

AA-CLIP là **Zero-Shot AD**: model KHÔNG được nhìn thấy test dataset trong lúc train.
Tác giả dùng chiến lược **train chéo dataset**:

```
TRAIN trên VisA  ──→  TEST zero-shot trên MVTec, BTAD, MPDD, Brain, Liver, Colon...
TRAIN trên MVTec ──→  TEST zero-shot trên VisA
```

**Notebook này thực hiện: Train VisA → Test MVTec** (tái tạo Table 1 trong paper)

### Hyperparameters theo paper (Implementation Details + Appendix A.3)

| Param | Giá trị | Nguồn |
|---|---|---|
| Backbone | ViT-L/14-336 (OpenCLIP) | Main paper |
| Image size | 518×518 | Main paper |
| K_T (text layers) | 3 | Main paper |
| K_I (image layers) | 6 | Main paper |
| λ, γ | 0.1 | Main paper |
| Stage 1 text_batch_size | **16** | Appendix A.3 |
| Stage 2 image_batch_size | **2** | Appendix A.3 |
| Stage 1 LR | 1×10⁻⁵, 5 epochs | Main paper |
| Stage 2 LR | 5×10⁻⁴, 20 epochs | Main paper |
| Adam β₁, β₂ | 0.5, 0.999 | Appendix A.3 |
| GPU (paper) | RTX 3090 24GB | Main paper |
| GPU (ta dùng) | Tesla P100 16GB | Kaggle |

> ⚠️ **Lưu ý P100 16GB:** text_batch_size=16 ổn (Stage 1 dùng no_grad cho image).
> Nếu Stage 2 OOM, giảm image_batch_size từ 2 xuống 1.

---
## 🗂️ Dataset cần Add vào notebook này

Vào **"Add data"** → tìm từng dataset → Add:

| Dataset | Vai trò | Kaggle ID | Mount path |
|---|---|---|---|
| **VisA** | TRAIN | `tensura3607/amazon-visa-anomaly` | `/kaggle/input/amazon-visa-anomaly/` |
| **MVTec AD** | TEST | `ipythonx/mvtec-ad` | `/kaggle/input/mvtec-ad/` |

## Cell 1: Setup — Clone, Fix bugs, Cài thư viện

In [ ]:
import os

# 1. Clone repo (bản đã refactor có config.yaml)
!git clone https://github.com/duonguwu/deeplearning_AACLIP.git
%cd deeplearning_AACLIP/AA-CLIP

# 2. Fix 'import ipdb' → crash ngay khi import trên Kaggle
for fname in ['train.py', 'forward_utils.py']:
    with open(fname, 'r') as f:
        code = f.read()
    code = code.replace('import ipdb\n', '# import ipdb  # disabled\n')
    with open(fname, 'w') as f:
        f.write(code)
print('✅ Fixed: ipdb imports')

# 3. Fix bug resume logic trong train.py
#    Bug: not(epoch == text_epoch-1) → sai khi resume từ checkpoint
#    Fix: epoch < text_epoch
with open('train.py', 'r') as f:
    code = f.read()
code = code.replace(
    'adapt_text = not (text_start_epoch == (args.text_epoch - 1))',
    'adapt_text = text_start_epoch < args.text_epoch  # bugfix'
)
with open('train.py', 'w') as f:
    f.write(code)
print('✅ Fixed: epoch resume logic')

# 4. Cài thư viện
!pip install -q open_clip_torch pyyaml ftfy kornia==0.6.9 timm==0.6.12 scikit-learn pandas
print('✅ Dependencies installed')

## Cell 2: Cấu hình Dataset Paths (config.yaml)

**VisA = dataset dùng để TRAIN** (mount tại `/kaggle/input/amazon-visa-anomaly/VisA_20220922/`)

**MVTec = dataset dùng để TEST** (mount tại `/kaggle/input/mvtec-ad/`)

Paths đã điền sẵn cho 2 Kaggle dataset IDs trên — không cần sửa nếu dùng đúng 2 dataset đó.

In [ ]:
import yaml, os

BASE_PATH    = '/kaggle/input'

# tensura3607/amazon-visa-anomaly → /kaggle/input/amazon-visa-anomaly/VisA_20220922/
VISA_FOLDER  = 'amazon-visa-anomaly/VisA_20220922'

# ipythonx/mvtec-ad → /kaggle/input/mvtec-ad/ (categories nằm trực tiếp ở root)
MVTEC_FOLDER = 'mvtec-ad'

config = {
    'paths': {
        'base_path': BASE_PATH,
        'datasets': {
            'Brain':          'MedAD/Brain_AD',
            'Liver':          'MedAD/Liver_AD',
            'Retina':         'MedAD/Retina_RESC_AD',
            'Colon_clinicDB': 'Colon/CVC-ClinicDB',
            'Colon_colonDB':  'Colon/CVC-ColonDB',
            'Colon_cvc300':   'Colon/CVC-300',
            'Colon_Kvasir':   'Colon/Kvasir',
            'BTAD':           'BTech_Dataset_transformed',
            'MPDD':           'MPDD',
            'MVTec':          MVTEC_FOLDER,
            'VisA':           VISA_FOLDER,
        }
    }
}

with open('config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print('✅ config.yaml đã tạo xong')
print(f'   [TRAIN] VisA  : {BASE_PATH}/{VISA_FOLDER}')
print(f'   [TEST]  MVTec : {BASE_PATH}/{MVTEC_FOLDER}')

## Cell 3: Verify GPU & Dataset Paths

Chạy cell này trước. **VisA (TRAIN) và MVTec (TEST) phải hiện `✅`** mới tiếp tục.

In [ ]:
import os, yaml

print('GPU INFO:')
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

print('\nDATASET PATHS CHECK:')
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

base = cfg['paths']['base_path']
critical = ['MVTec', 'VisA']  # Hai dataset bắt buộc phải có
all_critical_ok = True

for name, sub in cfg['paths']['datasets'].items():
    full = os.path.join(base, sub)
    ok   = os.path.exists(full)
    role = '[TRAIN]' if name == 'VisA' else '[TEST] ' if name == 'MVTec' else '       '
    tag  = '✅' if ok else '❌'
    print(f'  {role} {name:<20s}: {tag}  {full}')
    if name in critical and not ok:
        all_critical_ok = False

print()
if all_critical_ok:
    print('✅ VisA (TRAIN) + MVTec (TEST) đều OK — Ready!')
else:
    print('❌ Thiếu dataset quan trọng! Kiểm tra lại "Add data".')

print('\nCấu trúc bên trong VisA (train):')
!ls /kaggle/input/amazon-visa-anomaly/VisA_20220922/ 2>/dev/null | head -6
print('Cấu trúc bên trong MVTec (test):')
!ls /kaggle/input/mvtec-ad/ 2>/dev/null | head -6

---
## 4. Training: Cross-Dataset Zero-Shot Protocol

```
        ┌─────────────────────────────────────────────────────┐
        │  TRAIN trên VisA (2162 samples: 962 normal + 1200 anomaly)
        │     ↓ Stage 1: Text Adapter (5 epochs)
        │     ↓ Stage 2: Image Adapter (20 epochs)
        │  TEST zero-shot trên MVTec (model chưa thấy MVTec)
        └─────────────────────────────────────────────────────┘
```

**Tại sao P100 16GB chịu được text_batch_size=16?**
Stage 1 compute image features bằng `torch.no_grad()` → không lưu gradient cho image
→ Memory chủ yếu từ text adapter (nhỏ) → P100 16GB hoàn toàn đủ.

### Stage 1: Huấn luyện Text Adapter (train trên VisA)

Text Adapter học cách biểu diễn `Normal` vs `Anomaly` từ dữ liệu VisA.
Sau đó trọng số này sẽ được dùng **zero-shot** để test trên MVTec.

| Hyperparameter | Giá trị (paper) |
|---|---|
| Epochs | 5 |
| Learning Rate | 1×10⁻⁵ |
| **text_batch_size** | **16** (đúng theo Appendix A.3) |
| Optimizer | Adam (β₁=0.5, β₂=0.999) |

> ⏱️ Ước tính: ~30–60 phút trên P100 16GB (VisA full-shot 2162 samples)

In [ ]:
import os

TRAIN_DATASET    = 'VisA'   # Train trên VisA (cross-dataset protocol)
SAVE_PATH        = f'/kaggle/working/aaclip_ckpt/visa_trained'
TEXT_BATCH_SIZE  = 16       # Đúng theo paper Appendix A.3
IMAGE_BATCH_SIZE = 2        # Đúng theo paper Appendix A.3

os.makedirs(SAVE_PATH, exist_ok=True)
print(f'Train dataset : {TRAIN_DATASET}')
print(f'Save path     : {SAVE_PATH}')
print(f'text_batch    : {TEXT_BATCH_SIZE}  (paper default)')
print(f'image_batch   : {IMAGE_BATCH_SIZE}  (paper default)')
print('Bắt đầu Stage 1: Train Text Adapter (5 epochs)...')
print('=' * 60)

# image_epoch=0 → chỉ train Text Adapter, Image Adapter loop không chạy
!python train.py \
    --dataset {TRAIN_DATASET} \
    --training_mode full_shot \
    --save_path {SAVE_PATH} \
    --text_epoch 5 \
    --image_epoch 0 \
    --text_lr 0.00001 \
    --text_batch_size {TEXT_BATCH_SIZE} \
    --image_batch_size {IMAGE_BATCH_SIZE} \
    --text_adapt_until 3 \
    --image_adapt_until 6 \
    --text_norm_weight 0.1 \
    --text_adapt_weight 0.1 \
    --image_adapt_weight 0.1 \
    --img_size 518 \
    --seed 111

print('\n✅ Stage 1 hoàn thành!')
!ls -lh {SAVE_PATH}/text_adapter.pth

### Stage 2: Huấn luyện Image Adapter (tiếp tục trên VisA)

Image Adapter căn chỉnh patch features với text anchors đã học ở Stage 1.
Backbone **đóng băng hoàn toàn**.

| Hyperparameter | Giá trị (paper) |
|---|---|
| Epochs | 20 |
| Learning Rate | 5×10⁻⁴ |
| **image_batch_size** | **2** (đúng theo Appendix A.3) |
| LR Scheduler | MultiStepLR ×0.5 tại step 16k, 32k |

> ⚠️ Nếu P100 bị OOM ở Stage 2: giảm image_batch_size xuống **1**

> ⏱️ Ước tính: ~4–6 giờ trên P100 16GB (VisA full-shot, 20 epochs)

In [ ]:
import os

TRAIN_DATASET    = 'VisA'
SAVE_PATH        = '/kaggle/working/aaclip_ckpt/visa_trained'
TEXT_BATCH_SIZE  = 16
IMAGE_BATCH_SIZE = 2   # ← Giảm xuống 1 nếu P100 bị OOM

text_ckpt = f'{SAVE_PATH}/text_adapter.pth'
if not os.path.exists(text_ckpt):
    print('❌ Không tìm thấy text_adapter.pth! Chạy Stage 1 trước.')
else:
    print(f'✅ text_adapter.pth found')
    print('Bắt đầu Stage 2: Train Image Adapter (20 epochs)...')
    print('=' * 60)

    # text_epoch=5 → code load text_adapter.pth, kiểm tra (5 < 5 = False)
    # → bỏ qua text training → chỉ train Image Adapter
    !python train.py \
        --dataset {TRAIN_DATASET} \
        --training_mode full_shot \
        --save_path {SAVE_PATH} \
        --text_epoch 5 \
        --image_epoch 20 \
        --text_lr 0.00001 \
        --image_lr 0.0005 \
        --text_batch_size {TEXT_BATCH_SIZE} \
        --image_batch_size {IMAGE_BATCH_SIZE} \
        --text_adapt_until 3 \
        --image_adapt_until 6 \
        --text_norm_weight 0.1 \
        --text_adapt_weight 0.1 \
        --image_adapt_weight 0.1 \
        --img_size 518 \
        --seed 111

    print('\n✅ Stage 2 hoàn thành!')
    print('Checkpoints:')
    !ls -lh {SAVE_PATH}/image_adapter*.pth | tail -5

---
## 5. Evaluation: Test Zero-Shot trên MVTec

Model đã train trên VisA — giờ test **zero-shot** trên MVTec.
Model chưa bao giờ thấy ảnh MVTec trong lúc train.

`test.py` sẽ load từng `image_adapter_{epoch}.pth` (epoch 1→20),
chạy inference trên 15 class của MVTec, tính Image/Pixel AUROC+AP.

In [ ]:
TRAIN_DATASET = 'VisA'   # Checkpoint được train từ đây
TEST_DATASET  = 'MVTec'  # Test zero-shot trên MVTec
SAVE_PATH     = '/kaggle/working/aaclip_ckpt/visa_trained'

print(f'Train src : {TRAIN_DATASET}')
print(f'Test on   : {TEST_DATASET}  (zero-shot)')
print('=' * 60)

!python test.py \
    --dataset {TEST_DATASET} \
    --save_path {SAVE_PATH} \
    --img_size 518 \
    --text_adapt_until 3 \
    --image_adapt_until 6 \
    --text_adapt_weight 0.1 \
    --image_adapt_weight 0.1 \
    --batch_size 8 \
    --seed 111

print('\n✅ Evaluation xong!')
print(f'Log: {SAVE_PATH}/test.log')

## 6. Kết quả & So sánh với Paper (Table 1)

Lấy kết quả epoch cuối (epoch 20) và so sánh với paper.

In [ ]:
import pandas as pd

SAVE_PATH = '/kaggle/working/aaclip_ckpt/visa_trained'
LOG_PATH  = f'{SAVE_PATH}/test.log'

with open(LOG_PATH) as f:
    log = f.read()

# Lấy block kết quả của epoch cuối
blocks = log.split('load model from epoch')
print(f'Tổng số epochs đã eval: {len(blocks)-1}')
print('\n=== KẾT QUẢ EPOCH 20 (cuối) ===')
lines = [l.strip() for l in blocks[-1].split('\n') if l.strip()]
for l in lines:
    print(l)

print('\n=== SO SÁNH VỚI PAPER ===')
print('Protocol: Train VisA → Test MVTec zero-shot (Table 1, Full-shot)')
print()
# Số liệu từ Table 1 paper (full-shot, VisA→MVTec zero-shot)
# Xác nhận lại từ paper Table 1 cột MVTec, hàng AA-CLIP full-shot
comparison = pd.DataFrame({
    'Metric': ['Image AUROC', 'Image AP', 'Pixel AUROC', 'Pixel AP'],
    'Paper (CVPR 2025)': ['~91.1%', '~95.6%', '~56.2%', '~30.7%'],
    'Ours': ['→ xem log', '→ xem log', '→ xem log', '→ xem log'],
})
print(comparison.to_string(index=False))
print()
print('* Số liệu paper: Table 1, zero-shot, full-shot, MVTec AD.')
print('* Xác nhận lại giá trị chính xác từ file paper (Table 1).')
print('* Sai lệch <1% AUROC giữa P100 và RTX3090 là bình thường.')

## 7. Lưu Checkpoint (Download về máy)

Kaggle xóa `/kaggle/working` sau session. Zip lại để download.

In [ ]:
SAVE_PATH = '/kaggle/working/aaclip_ckpt/visa_trained'
ZIP_OUT   = '/kaggle/working/aaclip_visa_trained_checkpoints.zip'

!zip -r {ZIP_OUT} {SAVE_PATH}/
print(f'✅ Đã zip: {ZIP_OUT}')
!ls -lh {ZIP_OUT}